# Local rain-arrival model — 0 to 60 minutes

Predict a distribution over twelve five-minute arrival windows and “no rain within 60 minutes” at a currently dry location. This notebook prepares data, trains a small CNN + ConvGRU, evaluates baselines, calibrates probabilities and exports a self-describing checkpoint.

Select the project's `.venv` kernel. Run cells in order. **Training is opt-in** via `RUN_TRAINING`; no training or bot deployment occurs just by opening this file. The first data-preparation pass builds reusable local caches. Use the companion `rain_arrival_experiments.ipynb` for controlled comparisons.

This model does not generate a predicted radar image or predict rain ending. Those production capabilities remain separate.

In [ ]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import sys, json
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/data_processing").exists())
if str(PROJECT_ROOT / "src") not in sys.path: sys.path.insert(0, str(PROJECT_ROOT / "src"))
from arrival.data import (DataConfig, FrameStore, LabelStore, ArrivalDataset, prepare,
                          fixed_locations, CLASSES, CHANNELS, fingerprint)
from arrival.models import ArrivalNet
from arrival.training import (seed_everything, fit, predict, load_model, metrics,
                              fit_calibrator, calibrated_probs, persistence, motion_baseline)
from arrival.plots import plot_history, plot_evaluation, plot_example
SEED = 67
seed_everything(SEED,deterministic=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE, "| torch:", torch.__version__)
print("Channel order:", CHANNELS)

## Data contract and leakage controls

- Source radar uses the exact 33-color decoder; values are ordinal, **not mm/hour**. The existing conservative radar cleaner remains; no blanket removal of 1–5-pixel targets.
- Crops are `[T, 7, 64, 64]`. The target is pixel `[32,32]`; boundary locations without a full crop are excluded. Channel order is radar, temperature, humidity, wind_u, wind_v, station mask, distance. No explicit coordinates are input, but environmental fields can still reveal geography.
- Currently dry means fewer than five of the center 25 pixels exceed 0.01. First future qualifying observation maps to class 0–11; class 12 requires **all twelve** future observations to remain dry. Missing future data never means no rain.
- For T=6, the required interval is t−25 to t+60 (18 frames, 85 minutes). Shared manifests conservatively reserve T=12 history: t−55 to t+60, keeping anchors identical across history ablations.
- Whole days are partitioned chronologically before constructing anchors; complete windows remain inside one partition. Development ends before the previous held-out period. Train/validation/calibration are separate; a **new** final-test period is disabled until explicitly configured.
- Weather uses the exact frame-time CSV and rejects observation timestamps later than that frame. Historical publication/ingestion timestamps are unavailable, so this is observation-time causality, not measured historical availability.
- Training selects one of four arrival groups with equal probability globally, then an eligible training anchor and location. Expected group frequencies are 25% each; individual batches vary. Validation/calibration locations are fixed and sampled without arrival balancing. Spatial/event correlation remains.

In [ ]:
# Shared across both notebooks. Use a NEW directory when changing label/split definitions.
DATA = DataConfig(crop=64, max_history=12, future=12, patch=5,
                  rain_threshold=0.01, coverage=0.20, stride=1,
                  development_end="2026-08-26", test_start=None, seed=SEED)
DATA_DIR = PROJECT_ROOT / "models/arrival/data_v1"
store = FrameStore(PROJECT_ROOT)
label_store = LabelStore(store, DATA)
# First run may take time: exact source decoding, labels and causal weather caches.
# No API or SQL access. No existing radar-frame checkpoints/caches are overwritten.
prepared = prepare(store, DATA, DATA_DIR, normalization_frames=512)
manifests, norm = prepared["manifests"], prepared["norm"]
print({split: len(anchors) for split, anchors in manifests.items()})
print(prepared["caveat"])
print("Manifest hash:", prepared["manifest_hash"])

In [ ]:
counts = pd.DataFrame({k:v for k,v in prepared["class_counts"].items() if v is not None}, index=CLASSES)
display(counts)
counts.div(counts.sum()).plot.bar(figsize=(13,4), title="Natural eligible-location class frequencies (correlated samples)")
plt.ylabel("Fraction"); plt.tight_layout(); plt.show()

In [ ]:
# Seeded uniform anchor subset and natural-frequency spatial sampling for quick evaluation.
# Set EVALUATION_ANCHORS=None for all eligible anchors. Keep this fixed across experiments.
EVALUATION_ANCHORS = 256
EVALUATION_LOCATIONS = 16
def evaluation_records(split):
    anchors = manifests[split]
    if EVALUATION_ANCHORS is not None and len(anchors) > EVALUATION_ANCHORS:
        rng = np.random.default_rng(SEED)
        anchors = sorted(rng.choice(anchors, EVALUATION_ANCHORS, replace=False).tolist())
    return fixed_locations(label_store, anchors, EVALUATION_LOCATIONS, SEED)
validation_records = evaluation_records("validation")
calibration_records = evaluation_records("calibration")
(DATA_DIR / "evaluation_locations.json").write_text(json.dumps({
    "validation": validation_records, "calibration": calibration_records}, indent=2))
print("Fixed validation/calibration examples:", len(validation_records), len(calibration_records))
def make_dataset(history=6, channels=tuple(range(7)), split="train"):
    records = None if split == "train" else (validation_records if split == "validation" else calibration_records)
    return ArrivalDataset(store, label_store, manifests[split], norm, history, channels,
                          per_anchor=32, seed=SEED, records=records)

In [ ]:
train_data = make_dataset()
validation_data = make_dataset(split="validation")
calibration_data = make_dataset(split="calibration")
x, y = train_data[0]
assert tuple(x.shape) == (6, 7, 64, 64)
print("Training examples/epoch:", len(train_data), "| example target:", CLASSES[y])
fig, axes = plt.subplots(1,6,figsize=(15,3))
for t, ax in enumerate(axes):
    ax.imshow(x[t,0],vmin=0,vmax=1,cmap="viridis"); ax.scatter(32,32,c="magenta",marker="+")
    ax.set_title(f"t{(t-5)*5:+} min"); ax.axis("off")
plt.show()

## Baselines on the same natural-frequency subset

Persistence predicts no arrival because all eligible targets are currently dry. The motion baseline estimates one global translation from two full-domain frames; it is a transparent advection baseline, **not optical flow**. It cannot model growth/decay or nonuniform motion, and uses zero-filled domain boundaries. Use a fixed subset to bound runtime; compare learned predictions on this same subset later.

In [ ]:
BASELINE_ANCHORS = 64
unique_anchors = sorted({r[0] for r in validation_records})
rng = np.random.default_rng(SEED)
baseline_keys = set(rng.choice(unique_anchors, min(BASELINE_ANCHORS,len(unique_anchors)), replace=False))
baseline_records = [r for r in validation_records if r[0] in baseline_keys]
baseline_truth = np.array([label_store.get(k)[y,x] for k,y,x in baseline_records])
persistence_probs = persistence(baseline_records)
motion_probs, motion_shifts = motion_baseline(store, baseline_records, DATA)
baseline_results = {"persistence": metrics(persistence_probs,baseline_truth),
                    "global_translation": metrics(motion_probs,baseline_truth)}
display(pd.DataFrame({k:{m:v[m] for m in ["accuracy","macro_f1_all_13","missed_arrivals","false_arrivals"]}
                      for k,v in baseline_results.items()}).T)
(DATA_DIR / "baselines.json").write_text(json.dumps(baseline_results,indent=2))

## Train or load the model

The default head is 13-way classification, trained with negative log likelihood (equivalent to cross-entropy on logits). Mean validation average precision across 15/30/60 minutes selects the checkpoint. The lowest-NLL checkpoint is saved separately as best_nll.pt. The v5 run uses up to 30,000 updates, stopping after 40 checks without ranking improvement. Training loss uses balanced sampling while validation uses natural frequencies, so their absolute levels need not match. Use a fresh run name; existing checkpoints are never overwritten by a new run.

In [ ]:
RUN_TRAINING = False  # Set True when ready for the full training run.
RUN_NAME = "gru_t6_all_seed67_v5"
RUN_DIR = PROJECT_ROOT / "models/arrival/runs" / RUN_NAME
UPDATE_BUDGET = 30000
model = None
if RUN_TRAINING:
    seed_everything(SEED)
    model = ArrivalNet(history=6, channels=7, kind="gru", output="categorical").to(DEVICE)
    history = fit(model, train_data, validation_data, RUN_DIR, budget=UPDATE_BUDGET,
                  eval_every=250, patience=40, batch=16, seed=SEED, selection="mean_ap")
    plot_history(history)
elif (RUN_DIR / "best.pt").exists():
    model, checkpoint = load_model(RUN_DIR / "best.pt", DEVICE)
    if checkpoint["validation_records_hash"] != fingerprint(validation_records):
        raise ValueError("Checkpoint uses different evaluation records")
    if checkpoint["norm"] != norm: raise ValueError("Checkpoint normalization differs from prepared data")
    plot_history(json.loads((RUN_DIR / "history.json").read_text()))
else:
    print("Data is ready. Set RUN_TRAINING = False above and rerun that cell.")

In [ ]:
if model is not None:
    validation_log_probs, validation_truth = predict(model, validation_data)
    raw_probs = calibrated_probs(validation_log_probs)
    raw_metrics = metrics(raw_probs, validation_truth)
    display(pd.DataFrame(raw_metrics["horizons"]).T)
    print("Macro F1:",raw_metrics["macro_f1_all_13"],"| conditional time MAE:",raw_metrics["arrival_mae_minutes_detected"])
    print("Missed / actual arrivals:",raw_metrics["missed_arrivals"],"/",raw_metrics["rainy_cases"])
    plot_evaluation(raw_probs,validation_truth,raw_metrics)
    plot_example(validation_data,0,raw_probs[0])
    subset = ArrivalDataset(store,label_store,manifests["validation"],norm,records=baseline_records)
    lp, yt = predict(model,subset)
    baseline_results["gru_same_subset"] = metrics(calibrated_probs(lp),yt)
    display(pd.DataFrame({k:{m:v[m] for m in ["accuracy","macro_f1_all_13","missed_arrivals","false_arrivals"]}
                          for k,v in baseline_results.items()}).T)

    from arrival.ranking import plot_ranking, ranking_report
    # Compute explicitly: a running kernel may still hold the older metrics().
    raw_metrics['ranking'] = ranking_report(raw_probs, validation_truth)
    baseline_results['persistence']['ranking'] = ranking_report(persistence_probs, baseline_truth)
    baseline_results['global_translation']['ranking'] = ranking_report(motion_probs, baseline_truth)
    baseline_results['gru_same_subset']['ranking'] = ranking_report(calibrated_probs(lp), yt)
    plot_ranking(raw_metrics['ranking'])
    # Constant probabilities estimated exclusively from natural training labels.
    prior = np.asarray(prepared['class_counts']['train'], dtype=float)
    prior /= prior.sum()
    baseline_results['constant_training_prior'] = metrics(np.tile(prior, (len(yt),1)), yt)
    baseline_results['constant_training_prior']['ranking'] = ranking_report(np.tile(prior, (len(yt),1)), yt)
    comparison = []
    for name, result in baseline_results.items():
        for horizon, ranking in result['ranking'].items():
            comparison.append(dict(model=name, horizon=int(horizon),
                average_precision=ranking['average_precision'], prevalence=ranking['prevalence'],
                brier=result['horizons'][int(horizon)]['brier']))
    display(pd.DataFrame(comparison))
    (RUN_DIR / 'validation_metrics.json').write_text(json.dumps(raw_metrics,indent=2))
    (RUN_DIR / 'matched_baselines.json').write_text(json.dumps(baseline_results,indent=2))


## Calibrate probabilities on the separate natural-frequency partition

A shared increasing logistic map calibrates cumulative arrival probabilities using calibration data only. It preserves ranking at every horizon and ensures longer-horizon arrival probability never decreases. The previous class-bias calibration is retained as a comparison. This shared map is less flexible, so compare probability error as well as ranking. Calibration-fit scores are in-sample and are **not final performance claims**. Freeze the model/calibrator before evaluating a newly reserved test period. No-rain mistakes remain explicit; time MAE is only among true arrivals also predicted to arrive.

In [ ]:
calibrator = None
if model is not None:
    calibration_log_probs, calibration_truth = predict(model,calibration_data)
    calibrator = fit_calibrator(calibration_log_probs,calibration_truth)
    before = metrics(calibrated_probs(calibration_log_probs),calibration_truth)
    after_probs = calibrated_probs(calibration_log_probs,calibrator)
    after = metrics(after_probs,calibration_truth)
    display(pd.DataFrame({"before":before["horizons"],"after (fit partition)":after["horizons"]}))
    plot_evaluation(after_probs,calibration_truth,after)
    (RUN_DIR / "calibrator.json").write_text(json.dumps(calibrator,indent=2))
    (RUN_DIR / "calibration_fit_metrics.json").write_text(json.dumps({"before":before,"after":after},indent=2))
    probabilities = calibrated_probs(validation_log_probs[:1],calibrator)[0]
    display(pd.Series(probabilities,index=CLASSES,name="Example arrival distribution"))
    print({f"within_{minutes}_min":float(probabilities[:minutes//5].sum()) for minutes in (15,30,60)})

    # Full fixed validation sample, not just the 64-anchor baseline subset.
    # These are development scores: checkpoint selection already used validation.
    full_motion, _ = motion_baseline(store, validation_records, DATA)
    prior = np.asarray(prepared['class_counts']['train'], dtype=float)
    prior /= prior.sum()
    full_results = {
        'raw_gru': metrics(raw_probs, validation_truth),
        'calibrated_gru': metrics(calibrated_probs(validation_log_probs,calibrator), validation_truth),
        'motion': metrics(full_motion, validation_truth),
        'constant_training_prior': metrics(np.tile(prior,(len(validation_truth),1)),validation_truth),
    }
    rows=[]
    for name,result in full_results.items():
        for horizon,ranking in result['ranking'].items():
            scores=result['horizons'][int(horizon)]
            rows.append(dict(model=name,horizon=int(horizon),AP=ranking['average_precision'],
                             prevalence=ranking['prevalence'],**scores))
    display(pd.DataFrame(rows))
    (RUN_DIR/'full_validation_comparison.json').write_text(json.dumps(full_results,indent=2))
    plot_ranking(full_results['calibrated_gru']['ranking'])

    # Compare the previous calibration on the same development examples.
    from arrival.training import fit_legacy_calibrator
    legacy = fit_legacy_calibrator(calibration_log_probs, calibration_truth)
    legacy_result = metrics(calibrated_probs(validation_log_probs,legacy),validation_truth)
    (RUN_DIR/'legacy_calibration_comparison.json').write_text(json.dumps(legacy_result,indent=2))
    print('Legacy calibration AP:', {h:r['average_precision'] for h,r in legacy_result['ranking'].items()})
    # Threshold tradeoffs are development diagnostics, not chosen deployment settings.
    threshold_rows=[]
    calibrated_validation=calibrated_probs(validation_log_probs,calibrator)
    truth_array=np.asarray(validation_truth)
    for minutes in (15,30,60):
        probability=calibrated_validation[:,:minutes//5].sum(1)
        truth=truth_array<minutes//5
        for threshold in (.01,.02,.05,.10,.20,.30,.50):
            decision=probability>=threshold
            tp=int((decision & truth).sum()); fp=int((decision & ~truth).sum())
            threshold_rows.append(dict(horizon=minutes,threshold=threshold,
                precision=tp/max(tp+fp,1),recall=tp/max(int(truth.sum()),1),
                true_alerts=tp,false_alerts=fp,missed=int(truth.sum())-tp))
    display(pd.DataFrame(threshold_rows))
    (RUN_DIR/'validation_threshold_tradeoffs.json').write_text(json.dumps(threshold_rows,indent=2))

## Historical-only inference

The helper below reads only the anchor and its preceding history. It does not read arrival labels or future images. An already-raining target returns a separate status rather than an arrival forecast. The example reuses a known validation location for convenience; its inputs are still historical-only.

In [ ]:
from arrival.inference import predict_location
if model is not None:
    anchor, target_y, target_x = validation_records[0]
    forecast = predict_location(model,store,anchor,target_y,target_x,norm,DATA,calibrator)
    print(json.dumps(forecast,indent=2))

## Explicit final evaluation and export

`test_start=None` intentionally leaves the final test empty. Configure a genuinely new untouched period, prepare a **new data artifact directory**, keep development splits unchanged, and freeze all choices before enabling the test below. Do not reuse the older August/September test data already inspected in this project. An arrival model is not automatically connected to Telegram.

In [ ]:
RUN_FINAL_TEST = False
if RUN_FINAL_TEST:
    if model is None or calibrator is None or not DATA.test_start or not manifests["test"]:
        raise RuntimeError("Require frozen model/calibrator and a new complete test partition")
    test_records = fixed_locations(label_store,manifests["test"],EVALUATION_LOCATIONS,SEED)
    test_data = ArrivalDataset(store,label_store,manifests["test"],norm,records=test_records)
    lp, yt = predict(model,test_data)
    test_probs = calibrated_probs(lp,calibrator)
    result = metrics(test_probs,yt)
    (RUN_DIR / "final_test_metrics.json").write_text(json.dumps(result,indent=2))
    plot_evaluation(test_probs,yt,result)
if model is not None and calibrator is not None:
    export = dict(model_config=model.config,normalization=norm,data_config=vars(DATA),
                  channels=CHANNELS,classes=CLASSES,calibrator=calibrator,
                  manifest_hash=prepared["manifest_hash"],task="arrival_at_currently_dry_location")
    (RUN_DIR / "inference_config.json").write_text(json.dumps(export,indent=2))
    print("Saved best.pt, calibrator.json and inference_config.json in",RUN_DIR)

## Calibration experiment: independent curves with consistent horizons

Fits twelve cumulative logistic curves on calibration data, then applies least-squares isotonic projection across horizons. Tests probability error and ranking on held-out calibration dates and development validation. Projection prevents inconsistent horizons but may alter ranking. Uses the saved v5 checkpoint prediction caches; no training or deployment. Results are written to `reports/arrival_v5_calibration/projection_report.md`.

In [ ]:
# Standalone v5 experiment: uses cached predictions, never retrains the model.
import sys, subprocess
from pathlib import Path
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/arrival").exists())
result = subprocess.run([sys.executable, str(PROJECT_ROOT / "scripts/experiment_arrival_projection.py")],
                        cwd=PROJECT_ROOT, capture_output=True, text=True)
print(result.stdout)
if result.returncode:
    raise RuntimeError(result.stderr)
